# 01 — Data Inventory

Build both canonical parquet files and summarize all source data.
This notebook imports loaders from `src/loaders.py` and writes:
- `data/processed/panel_long.parquet`
- `data/processed/well_static.parquet`

In [4]:
import sys, os

# Ensure we're in the mari_poc root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
elif 'mari_poc' not in os.getcwd():
    os.chdir('mari_poc')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.loaders import (
    load_rates, load_pressures_whfp, load_bhp, load_bhp_pressure_xlsx,
    load_gas_gravity, load_gas_gravity_pressure_xlsx, load_subsurface
)
from src.config import (
    HORIZONTAL_WELLS, FORECAST_TARGETS, EXCLUDED_WELLS,
    PROCESSED_DIR, ALL_WELLS
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Loaders imported successfully.')

Loaders imported successfully.


## Load all source files

In [2]:
rates = load_rates()
whfp = load_pressures_whfp()
bhp = load_bhp()
bhp_xlsx = load_bhp_pressure_xlsx()
gg_csv = load_gas_gravity()
gg_xlsx = load_gas_gravity_pressure_xlsx()
sub = load_subsurface()

print('All source files loaded.')

All source files loaded.


## Source File Summary Table

In [4]:
summary_rows = []

# 1. BHP.csv
summary_rows.append({
    'File': 'BHP.csv',
    'Rows': len(bhp),
    'Columns': len(bhp.columns),
    'Date Range': f"{bhp['date'].min():%Y-%m} — {bhp['date'].max():%Y-%m}",
    'Wells': bhp['well'].nunique(),
    'Quality Notes': f"{bhp['bhp_psig'].isnull().sum()} null BHP (M-126H). Bare well names mapped."
})

# 2. gas_gravity.csv
summary_rows.append({
    'File': 'gas_gravity.csv',
    'Rows': len(gg_csv),
    'Columns': len(gg_csv.columns),
    'Date Range': 'N/A (static)',
    'Wells': gg_csv['well'].nunique(),
    'Quality Notes': 'Tab chars before values (stripped). Rounded to 2 dp. Already canonical names.'
})

# 3. STIXOR Sharing Data — Rates
summary_rows.append({
    'File': 'STIXOR Sharing Data.xlsx [Rates]',
    'Rows': len(rates),
    'Columns': len(rates.columns),
    'Date Range': f"{rates['date'].min():%Y-%m} — {rates['date'].max():%Y-%m}",
    'Wells': rates['well'].nunique(),
    'Quality Notes': f"choke nulls={rates['choke_64ths'].isnull().sum()}, water nulls={rates['water_bbl'].isnull().sum()}, gas nulls={rates['gas_mmcf'].isnull().sum()}"
})

# 4. STIXOR Sharing Data — Pressures
summary_rows.append({
    'File': 'STIXOR Sharing Data.xlsx [Pressures]',
    'Rows': len(whfp),
    'Columns': len(whfp.columns),
    'Date Range': f"{whfp['date'].min():%Y-%m} — {whfp['date'].max():%Y-%m}",
    'Wells': whfp['well'].nunique(),
    'Quality Notes': f"WHFP nulls={whfp['whfp_psig'].isnull().sum()}, line_pressure nulls={whfp['line_pressure_psig'].isnull().sum()}"
})

# 5. Subsurface data
summary_rows.append({
    'File': 'Subsurface data.xlsx',
    'Rows': len(sub),
    'Columns': len(sub.columns),
    'Date Range': 'N/A (static)',
    'Wells': sub['well'].nunique(),
    'Quality Notes': f"Skin nulls={sub['skin'].isnull().sum()} (all horizontals). Analog-copied values for 122H/124H/126H."
})

# 6. Pressure data xlsx (BHP)
summary_rows.append({
    'File': 'Pressure data.xlsx [Sheet2-BHP]',
    'Rows': len(bhp_xlsx),
    'Columns': 3,
    'Date Range': f"{bhp_xlsx['date'].min():%Y-%m} — {bhp_xlsx['date'].max():%Y-%m}",
    'Wells': bhp_xlsx['well'].nunique(),
    'Quality Notes': 'Multi-column layout. 21 wells (M-126H absent). Cross-check with BHP.csv.'
})

summary_df = pd.DataFrame(summary_rows)
summary_df

,File,Rows,Columns,Date Range,Wells,Quality Notes
0,BHP.csv,315,3,1988-07 — 2025-05,22,1 null BHP (M-126H). Bare well names mapped.
1,gas_gravity.csv,22,2,N/A (static),22,Tab chars before values (stripped). Rounded to...
2,STIXOR Sharing Data.xlsx [Rates],6652,6,1978-03 — 2025-06,22,"choke nulls=4561, water nulls=1969, gas nulls=30"
3,STIXOR Sharing Data.xlsx [Pressures],5354,4,1990-06 — 2025-06,22,"WHFP nulls=242, line_pressure nulls=3339"
4,Subsurface data.xlsx,22,9,N/A (static),22,Skin nulls=5 (all horizontals). Analog-copied ...
5,Pressure data.xlsx [Sheet2-BHP],314,3,1988-07 — 2025-05,21,Multi-column layout. 21 wells (M-126H absent)....


## Build panel_long.parquet

In [5]:
# Normalize dates to month-start
rates_m = rates.copy()
rates_m['date'] = rates_m['date'].dt.to_period('M').dt.to_timestamp()

whfp_m = whfp.copy()
whfp_m['date'] = whfp_m['date'].dt.to_period('M').dt.to_timestamp()

bhp_m = bhp.copy()
bhp_m['date'] = bhp_m['date'].dt.to_period('M').dt.to_timestamp()

# Deduplicate within each source
rates_m = rates_m.groupby(['well', 'date'], as_index=False).last()
whfp_m = whfp_m.groupby(['well', 'date'], as_index=False).last()
bhp_m = bhp_m.groupby(['well', 'date'], as_index=False).last()

# Merge
panel = rates_m.merge(whfp_m, on=['well', 'date'], how='outer')
panel = panel.merge(bhp_m, on=['well', 'date'], how='outer')

# Column order
panel = panel[['well', 'date', 'gas_mmcf', 'water_bbl', 'prod_days',
               'choke_64ths', 'whfp_psig', 'line_pressure_psig', 'bhp_psig']]
panel = panel.sort_values(['well', 'date']).reset_index(drop=True)

panel.to_parquet(PROCESSED_DIR / 'panel_long.parquet', index=False)
print(f'panel_long.parquet saved: {panel.shape[0]} rows × {panel.shape[1]} cols')
print(f'Wells: {panel["well"].nunique()}')
print(f'Date range: {panel["date"].min():%Y-%m} — {panel["date"].max():%Y-%m}')
print(f'\nNull counts:\n{panel.isnull().sum()}')

panel_long.parquet saved: 6865 rows × 9 cols
Wells: 22
Date range: 1978-03 — 2025-06

Null counts:
well                     0
date                     0
gas_mmcf               243
water_bbl             2182
prod_days             4773
choke_64ths           4774
whfp_psig             1753
line_pressure_psig    4850
bhp_psig              6551
dtype: int64


In [ ]:

# panel_check = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
# static = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

# panel_check.to_csv(PROCESSED_DIR / 'panel_long.csv', index=False)
# static.to_csv(PROCESSED_DIR / 'well_static.csv', index=False)


## Build well_static.parquet

In [6]:
gg = load_gas_gravity()
static = sub.merge(gg, on='well', how='left')
static['is_horizontal'] = static['well'].isin(HORIZONTAL_WELLS)

# Derive production timeline from panel
prod = panel.dropna(subset=['gas_mmcf'])
prod = prod[prod['gas_mmcf'] > 0]

first_prod = prod.groupby('well')['date'].min().rename('first_prod_date')
last_prod = prod.groupby('well')['date'].max().rename('last_prod_date')

dates_df = pd.DataFrame({'first_prod_date': first_prod, 'last_prod_date': last_prod})
dates_df['production_months'] = (
    (dates_df['last_prod_date'].dt.to_period('M').astype(int) -
     dates_df['first_prod_date'].dt.to_period('M').astype(int)) + 1
)
dates_df = dates_df.reset_index()

static = static.merge(dates_df, on='well', how='left')
static['production_months'] = static['production_months'].astype('Int64')

static.to_parquet(PROCESSED_DIR / 'well_static.parquet', index=False)
print(f'well_static.parquet saved: {static.shape[0]} rows × {static.shape[1]} cols')
print(f'\nNull counts:\n{static.isnull().sum()}')
print(f'\nFull table:')
static

well_static.parquet saved: 22 rows × 14 cols

Null counts:
well                 0
porosity             0
permeability_md      0
skin                 5
sw                   0
net_pay_m            0
chlorides_ppm        0
top_perf_md          0
bottom_perf_md       0
gas_gravity          0
is_horizontal        0
first_prod_date      0
last_prod_date       0
production_months    0
dtype: int64

Full table:


,well,porosity,permeability_md,skin,sw,net_pay_m,chlorides_ppm,top_perf_md,bottom_perf_md,gas_gravity,is_horizontal,first_prod_date,last_prod_date,production_months
0,M-11-HRL,0.20,7.4,-0.1,0.45,10.500000,17000,695.000000,705.500000,0.70,False,1978-03-01,2025-06-01,568
1,M-122H-HRL,0.22,34.0,NaN,0.46,530.000000,13000,1020.000000,1550.000000,0.77,True,2022-12-01,2025-06-01,31
2,M-123H-HRL,0.24,22.5,NaN,0.30,802.000000,1215,1044.000000,1846.000000,0.75,True,2023-11-01,2025-06-01,20
3,M-124H-HRL,0.22,34.0,NaN,0.46,475.000000,10000,918.000000,1393.000000,0.76,True,2023-12-01,2025-06-01,19
4,M-125H-HRL,0.24,22.5,NaN,0.30,758.000000,2500,951.000000,1709.000000,0.75,True,2024-09-01,2025-06-01,10
5,M-126H-HRL,0.22,34.0,NaN,0.46,719.000000,13000,993.000000,1712.000000,0.78,True,2024-10-01,2025-06-01,9
6,M-13-HRL,0.18,9.5,-2.8,0.31,10.200000,15000,691.600000,701.800000,0.69,False,1978-08-01,2025-06-01,563
7,M-22-HRL,0.19,5.9,5.3,0.42,11.650000,14000,708.750000,720.400000,0.77,False,1981-10-01,2025-06-01,525
8,M-41-HRL,0.17,37.1,-0.1,0.13,10.700000,13000,697.500000,708.200000,0.69,False,1986-04-01,2025-06-01,471
9,M-50-HRL,0.19,17.8,1.4,0.43,10.700000,15000,697.500000,708.200000,0.76,False,1986-04-01,2025-06-01,471


## Canonical Dataset Summary

In [8]:
# Reload from parquet to confirm they're valid
panel_check = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
static_check = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

ds_rows = []
ds_rows.append({
    'Dataset': 'panel_long.parquet',
    'Shape': f'{panel_check.shape[0]} × {panel_check.shape[1]}',
    'Date Range': f"{panel_check['date'].min():%Y-%m} — {panel_check['date'].max():%Y-%m}",
    'Wells': panel_check['well'].nunique(),
    'Null Cols': ', '.join(f"{c}={panel_check[c].isnull().sum()}" for c in panel_check.columns if panel_check[c].isnull().sum() > 0)
})
ds_rows.append({
    'Dataset': 'well_static.parquet',
    'Shape': f'{static_check.shape[0]} × {static_check.shape[1]}',
    'Date Range': f"{static_check['first_prod_date'].min():%Y-%m} — {static_check['last_prod_date'].max():%Y-%m}",
    'Wells': static_check['well'].nunique(),
    'Null Cols': ', '.join(f"{c}={static_check[c].isnull().sum()}" for c in static_check.columns if static_check[c].isnull().sum() > 0)
})

ds_df = pd.DataFrame(ds_rows)
ds_df

,Dataset,Shape,Date Range,Wells,Null Cols
0,panel_long.parquet,6865 × 9,1978-03 — 2025-06,22,"gas_mmcf=243, water_bbl=298, prod_days=4773, c..."
1,well_static.parquet,22 × 15,1978-03 — 2025-06,22,skin=5


## Observations

### Well-name oddities
- BHP.csv and Pressure xlsx Sheet2 use bare numbers (11, 122H, E-2) while all other files use M-XX-HRL form
- No truly ambiguous names found — the reconciliation map handles all variants cleanly

### Formatting surprises
- gas_gravity.csv has tab characters before numeric values
- STIXOR Sharing Data.xlsx has heavy leading whitespace in column names
- Subsurface xlsx has header on row 2 (rows 0-1 blank); net_pay column header contains a newline
- Pressure data xlsx Sheet2 uses a multi-column side-by-side layout (3 groups of well/date/pressure)
- Rates sheet 'Notes' column contains explanatory text in first few rows, then NaN

### Data quality flags
- **Analog-copied static properties**: M-122H/124H/126H share (φ=0.22, k=34, Sw=0.46); M-123H/125H share (φ=0.24, k=22.5, Sw=0.30)
- **M-123H chlorides = 1215 ppm** — an order of magnitude lower than all other wells (≥7000 ppm); possibly measured in a different unit or a data entry error
- **Horizontal net_pay (475–802 m)** is completed lateral length, not formation thickness — cannot be compared with vertical net_pay (6–15 m)
- **M-126H has no BHP in BHP.csv** (NaN) and is **absent from Pressure xlsx Sheet2 entirely**
- **Gas gravity discrepancy**: gas_gravity.csv is rounded to 2 dp; Pressure xlsx has 3+ dp precision. Max difference is 0.005 (4 wells). We use the csv as primary.
- **Line pressure very sparse** before ~2005 (3339/5354 nulls)
- **Choke and prod_days** not recorded before ~1993 (4560/6652 nulls)
- **Water_bbl** has 1969 nulls — many early months report gas only with no water column

### Columns dropped
- Rates: 'Notes' (metadata text), 'Unnamed: 7', 'Unnamed: 9'
- Subsurface: 'Unnamed: 0', 'Unnamed: 10', 'Unnamed: 11' (the latter had 'Notes:' label only)
- Pressure xlsx: 'Well Completion Sketch' sheet (embedded images, not tabular)

### Decisions requiring confirmation
1. **BHP source**: Used BHP.csv as primary (22 wells, 315 records). Pressure xlsx Sheet2 has 314 records, 21 wells (missing M-126H). The two sources appear to be the same data — confirm?
2. **Gas gravity source**: Used gas_gravity.csv (rounded). Pressure xlsx has slightly more precise values. Should we prefer the xlsx values?
3. **Water_bbl nulls**: Many months have null water production. Are these truly null (not measured) or should they be zero (measured, no water)? This matters for WGR computation.
4. **GWC datum**: Pressure xlsx Sheet2 note says '684 m TVD SS'. We add 70 m RT elevation → 754 m RKB. Confirm 70 m is correct RT for all wells, or does it vary?